# 🧠 Neural Network Chatbot — Powered by GPT (OpenAI API)

This notebook builds a working chatbot on top of OpenAI's GPT models — the same family of models that powers ChatGPT.

**What you'll build:**
- A Python class that talks to the OpenAI API and remembers conversation history
- A simple text-based chat loop that runs right in this notebook
- An optional web-style chat UI (using Gradio) you can share with a link

**What you need:**
- An OpenAI API key from https://platform.openai.com/api-keys
- An OpenAI account with billing set up (API usage is billed separately from a ChatGPT Plus subscription — new accounts get a small free credit, but you'll eventually need to add a card for continued use)

> Note: this notebook calls the *OpenAI API*, which is what actually powers ChatGPT. There's no separate "ChatGPT API" product — `chat.completions` is the correct endpoint people mean when they say that.

## Step 1 — Install the OpenAI Python SDK

In [ ]:
!pip install -q openai gradio


## Step 2 — Add Your API Key

Paste your key when prompted below. Using `getpass` means it's typed in securely and never printed or saved into the notebook file itself — this matters if you ever share this `.ipynb` with someone.

In [ ]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API key: ")
print("API key set for this session.")


## Step 3 — A Quick Word on "Neural Network"

GPT models are **transformer neural networks**: billions of parameters trained to predict the next word in a sequence, given everything said so far. When you call the API, you're sending the *entire conversation so far* as input, and the network predicts a reply one token at a time.

That's the core idea this whole notebook is built around — every time we "remember" the conversation below, we're really just re-sending the full history so the network has context, since the API itself is stateless between calls.

## Step 4 — The Chatbot Class

This class:
- Keeps a running conversation history (so the bot has memory within a session)
- Retries automatically if you hit a rate limit
- Tracks total tokens used, so you can keep an eye on cost

Pick your model based on your needs:
- `gpt-5.6-sol` — most capable, highest cost — best for complex reasoning
- `gpt-5.6-terra` — balanced quality/cost — good default for a chatbot (used below)
- `gpt-5.6-luna` — cheapest, fastest — good for high-volume/simple chat

Check https://platform.openai.com/docs/models for the current full list and pricing, since these change over time.

In [ ]:
from openai import OpenAI, APIError, RateLimitError, APIConnectionError
import time

client = OpenAI()  # automatically reads OPENAI_API_KEY from the environment

class ChatBot:
    def __init__(self, model="gpt-5.6-terra", system_prompt="You are a helpful, friendly assistant.", temperature=0.7):
        self.model = model
        self.temperature = temperature
        self.history = [{"role": "system", "content": system_prompt}]
        self.total_tokens = 0

    def send(self, user_message, max_retries=3):
        """Send a message, get a reply, and remember both in the conversation history."""
        self.history.append({"role": "user", "content": user_message})

        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model=self.model,
                    messages=self.history,
                    temperature=self.temperature,
                )
                reply = response.choices[0].message.content
                self.history.append({"role": "assistant", "content": reply})

                if response.usage:
                    self.total_tokens += response.usage.total_tokens

                return reply

            except RateLimitError:
                wait = 2 ** attempt
                print(f"Rate limited — retrying in {wait}s...")
                time.sleep(wait)
            except (APIConnectionError, APIError) as e:
                return f"⚠️ API error: {e}"

        return "⚠️ Failed after multiple retries. Please try again in a moment."

    def reset(self):
        """Clear conversation memory but keep the original system prompt."""
        system_prompt = self.history[0]
        self.history = [system_prompt]
        self.total_tokens = 0


## Step 5 — Chat In the Notebook (Text Loop)

Run this cell, then type in the box that appears. Type `reset` to clear memory, or `quit` to stop.

In [ ]:
bot = ChatBot(
    model="gpt-5.6-terra",
    system_prompt="You are a helpful, friendly assistant. Keep answers concise unless asked for detail."
)

print("🤖 Chatbot ready! Type 'quit' to stop, 'reset' to clear memory.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ("quit", "exit"):
        print(f"\n🤖 Goodbye! Total tokens used this session: {bot.total_tokens}")
        break

    if user_input.lower() == "reset":
        bot.reset()
        print("🤖 Memory cleared.\n")
        continue

    reply = bot.send(user_input)
    print(f"Bot: {reply}\n")


## Step 6 (Optional) — A Nicer Chat UI with Gradio

This gives you an actual chat window with a shareable public link (`share=True`), instead of typing into `input()` boxes. Great for a demo or viva.

In [ ]:
import gradio as gr

ui_bot = ChatBot(
    model="gpt-5.6-terra",
    system_prompt="You are a helpful, friendly assistant."
)

def respond(message, chat_history):
    reply = ui_bot.send(message)
    chat_history.append((message, reply))
    return "", chat_history

def clear_chat():
    ui_bot.reset()
    return []

with gr.Blocks(title="Neural Network Chatbot") as demo:
    gr.Markdown("# 🧠 Neural Network Chatbot\nBuilt on the OpenAI GPT API")
    chatbot_ui = gr.Chatbot(height=450)
    msg = gr.Textbox(placeholder="Ask me anything...", label="Your message")
    clear_btn = gr.Button("Clear conversation")

    msg.submit(respond, [msg, chatbot_ui], [msg, chatbot_ui])
    clear_btn.click(clear_chat, None, chatbot_ui)

demo.launch(share=True)


## Ways to Extend This Project

Good talking points for a report or viva:

- **Custom persona** — change `system_prompt` to make it a tutor, career counselor, FAQ bot for a specific domain, etc.
- **Streaming responses** — pass `stream=True` to `chat.completions.create` and print tokens as they arrive, for a "typing" effect.
- **Cost tracking** — `response.usage` gives `prompt_tokens`, `completion_tokens`, and `total_tokens`; multiply by the model's per-token price to log spend per conversation.
- **Retrieval-Augmented Generation (RAG)** — feed the bot your own documents (PDFs, notes) so it can answer questions grounded in that material instead of just its training data.
- **Voice** — add speech-to-text on input and text-to-speech on output for a voice assistant variant.
- **Persistence** — save `bot.history` to a file or database so conversations survive across sessions instead of resetting each run.

### Before you share this notebook
Colab doesn't save your typed API key into the `.ipynb` file itself, but it's good practice to go to **Runtime → Restart and run all**, then **Edit → Clear all outputs** before sharing, just to be safe.